In [ ]:
!pip -q install "pandas==2.2.2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 154.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [ ]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"
# MODEL_NAME = "intfloat/e5-large-v2"

embedder = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
print("Loaded:", MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded: BAAI/bge-large-en-v1.5


In [ ]:
tests = [
    # Same meaning, different phrasing (should be high)
    ("same_meaning_diff_phrasing",
     "The patient likely has bacterial pneumonia based on fever, cough, and focal consolidation on chest X-ray.",
     "Fever, productive cough, and a lobar opacity on CXR suggest bacterial pneumonia."),

    ("same_meaning_diff_phrasing",
     "Because troponin is rising and the ECG shows ST depression, treat this as NSTEMI and start antiplatelet therapy.",
     "Rising troponins plus ST depression points to NSTEMI, so initiate antiplatelet treatment."),

    # Similar phrasing, different meaning (should be lower)
    ("similar_phrasing_diff_meaning",
     "Start warfarin because the patient has atrial fibrillation and a high stroke risk.",
     "Start warfarin because the patient has active bleeding and a high stroke risk."),

    ("similar_phrasing_diff_meaning",
     "Give insulin to lower blood glucose and prevent ketoacidosis.",
     "Give glucose to lower blood glucose and prevent ketoacidosis."),

    # Negation flip (should drop)
    ("negation_flip",
     "This presentation is consistent with pulmonary embolism and should be treated promptly.",
     "This presentation is not consistent with pulmonary embolism and should not be treated as such."),

    # Dose flip (should drop)
    ("dose_flip",
     "Administer 5 mg of morphine IV for severe pain.",
     "Administer 50 mg of morphine IV for severe pain."),
]


In [ ]:
labels = [t[0] for t in tests]
A = [t[1] for t in tests]
B = [t[2] for t in tests]

embA = embedder.encode(A, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)
embB = embedder.encode(B, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)

sims = (embA * embB).sum(axis=1)  # cosine because normalized

df = pd.DataFrame({
    "label": labels,
    "cosine_similarity": sims,
    "A": A,
    "B": B,
}).sort_values("cosine_similarity", ascending=False)

df


,label,cosine_similarity,A,B
3,similar_phrasing_diff_meaning,0.947764,Give insulin to lower blood glucose and preven...,Give glucose to lower blood glucose and preven...
1,same_meaning_diff_phrasing,0.944077,Because troponin is rising and the ECG shows S...,Rising troponins plus ST depression points to ...
2,similar_phrasing_diff_meaning,0.939754,Start warfarin because the patient has atrial ...,Start warfarin because the patient has active ...
5,dose_flip,0.920673,Administer 5 mg of morphine IV for severe pain.,Administer 50 mg of morphine IV for severe pain.
0,same_meaning_diff_phrasing,0.826194,The patient likely has bacterial pneumonia bas...,"Fever, productive cough, and a lobar opacity o..."
4,negation_flip,0.732373,This presentation is consistent with pulmonary...,This presentation is not consistent with pulmo...


In [ ]:
df.groupby("label")["cosine_similarity"].agg(["count","mean","min","max"]).sort_values("mean", ascending=False)


,count,mean,min,max
label,,,,
similar_phrasing_diff_meaning,2,0.943759,0.939754,0.947764
dose_flip,1,0.920673,0.920673,0.920673
same_meaning_diff_phrasing,2,0.885136,0.826194,0.944077
negation_flip,1,0.732373,0.732373,0.732373


In [ ]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

TESTS = [
    ("same_meaning_diff_phrasing",
     "The patient likely has bacterial pneumonia based on fever, cough, and focal consolidation on chest X-ray.",
     "Fever, productive cough, and a lobar opacity on CXR suggest bacterial pneumonia."),

    ("same_meaning_diff_phrasing",
     "Because troponin is rising and the ECG shows ST depression, treat this as NSTEMI and start antiplatelet therapy.",
     "Rising troponins plus ST depression points to NSTEMI, so initiate antiplatelet treatment."),

    ("similar_phrasing_diff_meaning",
     "Start warfarin because the patient has atrial fibrillation and a high stroke risk.",
     "Start warfarin because the patient has active bleeding and a high stroke risk."),

    ("similar_phrasing_diff_meaning",
     "Give insulin to lower blood glucose and prevent ketoacidosis.",
     "Give glucose to lower blood glucose and prevent ketoacidosis."),

    ("negation_flip",
     "This presentation is consistent with pulmonary embolism and should be treated promptly.",
     "This presentation is not consistent with pulmonary embolism and should not be treated as such."),

    ("dose_flip",
     "Administer 5 mg of morphine IV for severe pain.",
     "Administer 50 mg of morphine IV for severe pain."),
]

MODELS = [
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-base-v2",
    "intfloat/e5-large-v2",      # comment out if too slow on CPU
    # "ncbi/MedCPT-Query-Encoder" # optional, can be slower
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def run_model(model_name):
    emb = SentenceTransformer(model_name, device=device)
    A = [t[1] for t in TESTS]
    B = [t[2] for t in TESTS]
    eA = emb.encode(A, batch_size=16, convert_to_numpy=True, normalize_embeddings=True)
    eB = emb.encode(B, batch_size=16, convert_to_numpy=True, normalize_embeddings=True)
    sims = (eA * eB).sum(axis=1)

    df = pd.DataFrame({
        "model": model_name,
        "label": [t[0] for t in TESTS],
        "cosine": sims
    })

    stats = df.groupby(["model","label"])["cosine"].agg(["count","mean","min","max"]).reset_index()
    return df, stats

all_stats = []
for m in MODELS:
    _, st = run_model(m)
    all_stats.append(st)

out = pd.concat(all_stats, ignore_index=True)
out.sort_values(["model","mean"], ascending=[True, False])


Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

,model,label,count,mean,min,max
3,BAAI/bge-base-en-v1.5,similar_phrasing_diff_meaning,2,0.959877,0.958933,0.960821
0,BAAI/bge-base-en-v1.5,dose_flip,1,0.934555,0.934555,0.934555
2,BAAI/bge-base-en-v1.5,same_meaning_diff_phrasing,2,0.888517,0.851015,0.926019
1,BAAI/bge-base-en-v1.5,negation_flip,1,0.791342,0.791342,0.791342
7,intfloat/e5-base-v2,similar_phrasing_diff_meaning,2,0.983290,0.981557,0.985023
4,intfloat/e5-base-v2,dose_flip,1,0.967450,0.967450,0.967450
6,intfloat/e5-base-v2,same_meaning_diff_phrasing,2,0.936475,0.913249,0.959702
5,intfloat/e5-base-v2,negation_flip,1,0.912773,0.912773,0.912773
11,intfloat/e5-large-v2,similar_phrasing_diff_meaning,2,0.982987,0.980990,0.984984
8,intfloat/e5-large-v2,dose_flip,1,0.961151,0.961151,0.961151


In [ ]:
!pip -q install -U sentence-transformers pandas numpy

# Optional (API)
!pip -q install -U openai tiktoken
!pip -q install -U voyageai


In [ ]:
tests = [
    ("same_meaning_diff_phrasing",
     "The patient likely has bacterial pneumonia based on fever, cough, and focal consolidation on chest X-ray.",
     "Fever, productive cough, and a lobar opacity on CXR suggest bacterial pneumonia."),

    ("similar_phrasing_diff_meaning",
     "Start warfarin because the patient has atrial fibrillation and a high stroke risk.",
     "Start warfarin because the patient has active bleeding and a high stroke risk."),

    ("negation_flip",
     "This presentation is consistent with pulmonary embolism and should be treated promptly.",
     "This presentation is not consistent with pulmonary embolism and should not be treated as such."),

    ("dose_flip",
     "Administer 5 mg of morphine IV for severe pain.",
     "Administer 50 mg of morphine IV for severe pain."),

    # long-ish reasoning style pair
    ("reasoning_paraphrase",
     "Because troponin is rising and the ECG shows ST depression, this is most consistent with NSTEMI; start antiplatelet therapy and evaluate for early invasive management.",
     "Rising troponins with ST depression indicates NSTEMI, so initiate antiplatelet treatment and consider an early invasive approach."),
]


In [ ]:
import re
import numpy as np

def split_sentences(text: str):
    # simple sentence split
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sents if s]

def chunk_by_char_budget(text: str, max_chars: int = 1200):
    # build chunks of sentences up to a character budget
    sents = split_sentences(text)
    chunks, cur = [], ""
    for s in sents:
        if len(cur) + len(s) + 1 <= max_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = s
    if cur:
        chunks.append(cur)
    return chunks

def mean_pool_normalized(embs: np.ndarray):
    v = embs.mean(axis=0)
    v = v / (np.linalg.norm(v) + 1e-12)
    return v


In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LOCAL_MODELS = [
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-base-v2",
    # If you later get GPU, try:
    "BAAI/bge-large-en-v1.5",
    "intfloat/e5-large-v2",
]

def embed_local_text(embedder, text: str, max_chars=1200):
    chunks = chunk_by_char_budget(text, max_chars=max_chars)
    embs = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True, batch_size=16)
    return mean_pool_normalized(embs)

rows = []
for model_name in LOCAL_MODELS:
    emb = SentenceTransformer(model_name, device=device)
    for label, a, b in tests:
        ea = embed_local_text(emb, a)
        eb = embed_local_text(emb, b)
        sim = float(np.dot(ea, eb))
        rows.append({"model": model_name, "label": label, "cosine": sim})

df = pd.DataFrame(rows)
display(df.pivot_table(index="label", columns="model", values="cosine", aggfunc="mean").sort_index())
display(df.groupby(["model","label"])["cosine"].agg(["count","mean","min","max"]).reset_index())


Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

model,BAAI/bge-base-en-v1.5,BAAI/bge-large-en-v1.5,intfloat/e5-base-v2,intfloat/e5-large-v2
label,,,,
dose_flip,0.934555,0.920673,0.967450,0.961151
negation_flip,0.791342,0.732373,0.912773,0.917820
reasoning_paraphrase,0.938750,0.946330,0.971776,0.970328
same_meaning_diff_phrasing,0.851015,0.826194,0.913248,0.922183
similar_phrasing_diff_meaning,0.960821,0.939754,0.981557,0.984984


,model,label,count,mean,min,max
0,BAAI/bge-base-en-v1.5,dose_flip,1,0.934555,0.934555,0.934555
1,BAAI/bge-base-en-v1.5,negation_flip,1,0.791342,0.791342,0.791342
2,BAAI/bge-base-en-v1.5,reasoning_paraphrase,1,0.938750,0.938750,0.938750
3,BAAI/bge-base-en-v1.5,same_meaning_diff_phrasing,1,0.851015,0.851015,0.851015
4,BAAI/bge-base-en-v1.5,similar_phrasing_diff_meaning,1,0.960821,0.960821,0.960821
5,BAAI/bge-large-en-v1.5,dose_flip,1,0.920673,0.920673,0.920673
6,BAAI/bge-large-en-v1.5,negation_flip,1,0.732373,0.732373,0.732373
7,BAAI/bge-large-en-v1.5,reasoning_paraphrase,1,0.946330,0.946330,0.946330
8,BAAI/bge-large-en-v1.5,same_meaning_diff_phrasing,1,0.826194,0.826194,0.826194
9,BAAI/bge-large-en-v1.5,similar_phrasing_diff_meaning,1,0.939754,0.939754,0.939754


In [ ]:
import os
from openai import OpenAI

# Set your key:
# os.environ["OPENAI_API_KEY"] = "..."

client = OpenAI()

def embed_openai_text(text: str, model="text-embedding-3-large", max_chars=1200):
    chunks = chunk_by_char_budget(text, max_chars=max_chars)
    resp = client.embeddings.create(model=model, input=chunks)
    embs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    # normalize each chunk embedding then mean-pool
    embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12)
    return mean_pool_normalized(embs)

rows = []
for label, a, b in tests:
    ea = embed_openai_text(a)
    eb = embed_openai_text(b)
    sim = float(np.dot(ea, eb))
    rows.append({"model": "openai:text-embedding-3-large", "label": label, "cosine": sim})

pd.DataFrame(rows).sort_values("cosine", ascending=False)


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [2]:
!pip -q install --upgrade --force-reinstall \
  "numpy==2.0.2" \
  "pandas==2.2.2" \
  "scipy==1.11.4" \
  "scikit-learn==1.5.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.0 MB/s eta 0:00:00
ERROR: Cannot install numpy==2.0.2, pandas==2.2.2 and scipy==1.11.4 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [3]:
!pip -q install "sentence-transformers==3.0.1" "transformers==4.44.2" "torch" --no-deps
!pip -q install "huggingface-hub" "tqdm" "regex"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 92.2 MB/s eta 0:00:00


In [4]:
# =========================
# Local (no-API) embedding models via Hugging Face / Sentence-Transformers
# =========================
# This adds local embedders to the same comparison framework:
# - BGE (general, strong)
# - E5 (general, strong)
# - A biomedical-focused option (BioLinkBERT via sentence-transformers if available)
#
# NOTE: If you're on Colab GPU, you'll get good speed.
# =========================


import os, re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------
# Template-shaped test cases (same as before, embed Explanation only)
# -------------------------
PROMPT_OUTPUT_CASES = [
    {
        "label": "pos_same_meaning_diff_phrasing",
        "teacher": """Decision: Unsafe
Explanation: This combination can increase bleeding risk through additive anticoagulant and antiplatelet effects. In older patients, the risk is higher due to reduced physiologic reserve and comorbidity burden. If renal function is impaired, drug exposure or metabolite accumulation can further increase harm. Monitor for signs of bleeding and consider alternative therapy or dose adjustments. The mechanism is primarily pharmacodynamic synergy with possible pharmacokinetic contributions depending on the specific agents.
Uncertainty notes: If the exact regimen and doses are unclear, confirm the indication and review recent labs before proceeding.""",
        "student": """Decision: Unsafe
Explanation: Using these agents together is generally unsafe because their effects on hemostasis can stack, leading to clinically significant bleeding. Advanced age and comorbid conditions increase vulnerability to adverse outcomes. Reduced kidney function can worsen risk by increasing systemic exposure in some drugs. Consider safer alternatives, minimize duration if unavoidable, and monitor closely for bleeding. The main issue is additive pharmacodynamic anticoagulation/antiplatelet activity, with possible PK impact in renal impairment.
Uncertainty notes: Verify doses, indication, and current labs (renal function, hemoglobin) to resolve uncertainty safely."""
    },
    {
        "label": "pos_same_meaning_diff_phrasing",
        "teacher": """Decision: Unsafe
Explanation: This combination may cause excessive CNS depression because both agents can impair alertness and respiratory drive. Hepatic dysfunction can reduce clearance and increase sedation, making toxicity more likely. The interaction is largely pharmacodynamic, but slower metabolism can amplify the effect. Avoid co-administration when possible and use the lowest effective dose with monitoring if unavoidable.
Uncertainty notes: If liver function severity is uncertain, review recent LFTs and consider specialist input.""",
        "student": """Decision: Unsafe
Explanation: Co-prescribing these drugs is risky because they can synergize to depress the central nervous system, increasing oversedation and respiratory suppression. In patients with impaired liver function, reduced clearance can further elevate drug levels and worsen toxicity. The mechanism is mainly PD synergy, while hepatic impairment can magnify the effect through slower metabolism. Prefer alternatives or reduce dosing with careful monitoring.
Uncertainty notes: Confirm degree of hepatic impairment via recent labs before finalizing therapy."""
    },
    {
        "label": "neg_similar_phrasing_diff_meaning",
        "teacher": """Decision: Unsafe
Explanation: This combination can increase bleeding risk through additive anticoagulant and antiplatelet effects. In older patients, the risk is higher due to reduced physiologic reserve and comorbidity burden. If renal function is impaired, drug exposure or metabolite accumulation can further increase harm. Monitor for signs of bleeding and consider alternative therapy or dose adjustments. The mechanism is primarily pharmacodynamic synergy with possible pharmacokinetic contributions depending on the specific agents.
Uncertainty notes: If the exact regimen and doses are unclear, confirm the indication and review recent labs before proceeding.""",
        "student": """Decision: Safe
Explanation: This combination can increase bleeding risk through additive anticoagulant and antiplatelet effects. In older patients, the risk is higher due to reduced physiologic reserve and comorbidity burden. If renal function is impaired, drug exposure or metabolite accumulation can further increase harm. Monitor for signs of bleeding and consider alternative therapy or dose adjustments. The mechanism is primarily pharmacodynamic synergy with possible pharmacokinetic contributions depending on the specific agents.
Uncertainty notes: If the exact regimen and doses are unclear, confirm the indication and review recent labs before proceeding."""
    },
    {
        "label": "neg_similar_phrasing_diff_meaning",
        "teacher": """Decision: Unsafe
Explanation: This combination may cause excessive CNS depression because both agents can impair alertness and respiratory drive. Hepatic dysfunction can reduce clearance and increase sedation, making toxicity more likely. The interaction is largely pharmacodynamic, but slower metabolism can amplify the effect. Avoid co-administration when possible and use the lowest effective dose with monitoring if unavoidable.
Uncertainty notes: If liver function severity is uncertain, review recent LFTs and consider specialist input.""",
        "student": """Decision: Unsafe
Explanation: This combination may cause excessive CNS depression because both agents can impair alertness and respiratory drive. Hepatic dysfunction can improve clearance and decrease sedation, making toxicity less likely. The interaction is largely pharmacodynamic, but faster metabolism can amplify the effect. Avoid co-administration when possible and use the lowest effective dose with monitoring if unavoidable.
Uncertainty notes: If liver function severity is uncertain, review recent LFTs and consider specialist input."""
    },
]

def parse_output(text: str):
    t = text.strip().replace("\r\n", "\n")
    m = re.search(r"(?im)^\s*Decision:\s*(Safe|Unsafe)\s*$", t)
    decision = m.group(1) if m else None

    m = re.search(r"(?is)^\s*Explanation:\s*(.*?)(?:^\s*Uncertainty notes:\s*|\Z)", t, re.M)
    explanation = m.group(1).strip() if m else ""

    m = re.search(r"(?is)^\s*Uncertainty notes:\s*(.*)\Z", t, re.M)
    uncertainty = m.group(1).strip() if m else ""

    return {"decision": decision, "explanation": explanation, "uncertainty": uncertainty}

def cosine(u, v):
    u = u / (np.linalg.norm(u) + 1e-12)
    v = v / (np.linalg.norm(v) + 1e-12)
    return float(u @ v)

# -------------------------
# Local embedder wrappers
# -------------------------
LOCAL_MODELS = {
    # Strong general-purpose (recommended)
    "local:bge-large-en-v1.5": "BAAI/bge-large-en-v1.5",
    "local:e5-large-v2": "intfloat/e5-large-v2",

    # Faster variants if needed
    # "local:bge-base-en-v1.5": "BAAI/bge-base-en-v1.5",
    # "local:e5-base-v2": "intfloat/e5-base-v2",

    # Biomedical-ish option (try; if it errors, skip it)
    # There are multiple biomedical sentence-transformer checkpoints; this is a common baseline:
    "local:biolinkbert-base": "michiyasunaga/BioLinkBERT-base",
}

# Build SentenceTransformer models
embedders = {}
for name, hf_id in LOCAL_MODELS.items():
    try:
        embedders[name] = SentenceTransformer(hf_id, device=device)
        print("Loaded:", name, "=>", hf_id)
    except Exception as e:
        print("SKIP:", name, "=>", hf_id, "| error:", str(e)[:200])

def embed_local(model: SentenceTransformer, text: str):
    # Encode as a single vector; normalize embeddings for cosine similarity
    v = model.encode([text], convert_to_numpy=True, normalize_embeddings=True, batch_size=8)[0]
    return v

# -------------------------
# Compare local embedders on Explanation similarity
# -------------------------
rows = []
for embedder_name, model in embedders.items():
    for ex in PROMPT_OUTPUT_CASES:
        T = parse_output(ex["teacher"])
        S = parse_output(ex["student"])
        vt = embed_local(model, T["explanation"])
        vs = embed_local(model, S["explanation"])
        sim = cosine(vt, vs)
        rows.append({
            "embedder": embedder_name,
            "label": ex["label"],
            "sim_explanation": sim,
            "teacher_decision": T["decision"],
            "student_decision": S["decision"],
            "decision_match": (T["decision"] == S["decision"]),
        })

df = pd.DataFrame(rows)
display(df)

summary = df.groupby(["embedder","label"])["sim_explanation"].mean().unstack()
display(summary)

sep = df.groupby("embedder").apply(
    lambda g: g[g["label"].str.startswith("pos")]["sim_explanation"].mean()
              - g[g["label"].str.startswith("neg")]["sim_explanation"].mean()
).rename("pos_minus_neg").to_frame()

display(sep.sort_values("pos_minus_neg", ascending=False))


ImportError: cannot import name 'cached_property' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)